# Linear Regression with Time Series Data

In [9]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import pytz
from pymongo import MongoClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

## Prepare Data

### Import

In [11]:
## Complete to the create a client to connect to the MongoDB server, assign the "air-quality" database to db, 
## and assign the "nairobi" connection to nairobi.

client = MongoClient(host ="localhost", port=27017)
db = client["air-quality"]
nairobi = db["nairobi"]

In [23]:
## Complete the wrangle function below so that the results from the database query are read into the DataFrame df. 
## Be sure that the index of df is the "timestamp" from the results.

def wrangle(collection):
    results = collection.find(
        {"metadata.site": 29, "metadata.measurement": "P2"},
        projection={"P2": 1, "timestamp": 1, "_id": 0},
    )

    df = pd.DataFrame(results).set_index("timestamp")
    
    # localise time zone
    df.index = df.index.tz_localize("UTC").tz_convert("Africa/Nairobi")
    # Remove outliers
    df= df[df["P2"] < 500]
    ## Resample to 1H window, ffill missing values
    df = df["P2"].resample("1H").mean().fillna(method="ffill").to_frame()
    
    # Add lag feature
    df["P2.L1"] = df["P2"].shift(1)
    # Drop NaN rows
    df.dropna(inplace=True)
    
    return df

In [ ]:
## Use your wrangle function to read the data from the nairobi collection into the DataFrame df.

df = wrangle(nairobi)
df.head()

In [ ]:
## Add to your wrangle function so that the DatetimeIndex for df is localized to the correct timezone, "Africa/Nairobi". 
## Don't forget to re-run all the cells above after you change the function.

df.index.tz_localize("UTC").tz_convert("Africa/Nairobi")[:5]

In [ ]:
# Create a boxplot of the "P2" readings in df.
fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].plot(kind="box", vert=False, title="Distribution of PM2, 5 Readings", ax =ax);

In [ ]:
# Add to your wrangle function so that all "P2" readings above 500 are dropped from the dataset. 
#Don't forget to re-run all the cells above after you change the function.

# Remove outliers
df= df[df["P2"] < 500]

In [ ]:
## Create a time series plot of the "P2" readings in df.
fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].plot(xlabel= "Time", ylabel= "PM2.5", title=" PM2.5 Readings", ax =ax);

In [ ]:
## Add to your wrangle function to resample df to provide the mean "P2" reading for each hour. Use a forward fill to impute any missing values. 
# Don't forget to re-run all the cells above after you change the function.

## Resample to 1H window, ffill missing values
df["P2"].resample("1H").mean().fillna(method="ffill").to_frame().head()

In [ ]:
### Plot the rolling average of the "P2" readings in df. Use a window size of 168 (the number of hours in a week).

fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].rolling(168).mean().plot(ax=ax,ylabel="PM2.5", title="Weekly Rolling Average");

In [ ]:
## Add to your wrangle function to create a column called "P2.L1" that contains the mean"P2" reading from the previous hour. 
## Since this new feature will create NaN values in your DataFrame, be sure to also drop null rows from df.

# Add lag feature
df["P2.L1"] = df["P2"].shift(1)
# Drop NaN rows
df.dropna(inplace=True).head()

In [ ]:
## Create a correlation matrix for df.

corr = df[['P2', 'P2.L1']].corr()
corr

In [ ]:
## Create a scatter plot that shows PM 2.5 mean reading for each our as a function of the mean reading from the previous hour. 
## In other words, "P2.L1" should be on the x-axis, and "P2" should be on the y-axis. Don't forget to label your axes!

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x=df["P2.L1"], y=df["P2"])
ax.plot([0, 120], [0, 120], linestyle="--", color="orange")
plt.xlabel("P2.L1")
plt.ylabel("P2")
plt.title("PM2.5 Autocorrelation")

## Split

In [ ]:
## Split the DataFrame df into the feature matrix X and the target vector y. Your target is "P2".

target = "P2"
y = df[target]
X = df.drop(columns=target)

In [ ]:
## Split X and y into training and test sets. The first 80% of the data should be in your training set. 
## The remaining 20% should be in the test set.

cutoff = int(len(X) * 0.8)

X_train, y_train = X.iloc[:cutoff], y.iloc[:cutoff] # start from the first observation till cutoff
X_test, y_test = X.iloc[cutoff:], y.iloc[cutoff:]


# Build Model

## Baseline

In [ ]:
### Calculate the baseline mean absolute error for your model.

y_pred_baseline =[y_train.mean()] * len(y_train)
mae_baseline =  mean_absolute_error(y_train, y_pred_baseline)

print("Mean P2 Reading:", round(y_train.mean(), 2))
print("Baseline MAE:", round(mae_baseline, 2))

## Iterate

In [ ]:
### instantiate a LinearRegression model named model, and fit it to your training data.

model = LinearRegression()
model.fit(X_train, y_train)


## Evaluate

In [ ]:
## Calculate the training and test mean absolute error for your model.

training_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))
print("Training MAE:", round(training_mae, 2))
print("Test MAE:", round(test_mae, 2))

# Communicate Results

In [ ]:
## Extract the intercept and coefficient from your model.

intercept = model.intercept_.round(2)
coefficient = model.coef_.round(2)[0]

print(f"P2 = {intercept} + ({coefficient} * P2.L1)")

In [ ]:
## Create a DataFrame df_pred_test that has two columns: "y_test" and "y_pred". The first should contain the true values for your test set, 
## and the second should contain your model's predictions. Be sure the index of df_pred_test matches the index of y_test.

df_pred_test = pd.DataFrame(
    {
        "y_test": y_test,
        "y_pred": model.predict(X_test)
    }
)
df_pred_test.head()

In [ ]:
## create a time series line plot for the values in test_predictions using plotly express. Be sure that the y-axis is properly labeled as "P2".
fig = px.line(df_pred_test, labels={"value":"P2"})
fig.show()